# الإنسان في الحلقة: بوابات ما قبل الإجراء، تصنيف المخاطر، وتسجيل التدقيق

الملف التعريفي لهذا الدرس يقدّم مفهوم الإنسان في الحلقة مع مقتطف قصير يطلب من المستخدم `APPROVE` أو `REJECT` بعد أن ينتج الوكيل ردًا. هذه النموذج نقطة انطلاق جيدة، لكن تطبيقات الإنسان في الحلقة في الإنتاج عادةً ما تحتاج إلى ثلاث قطع إضافية:

1. **بوابة ما قبل الإجراء** تعمل **قبل** أن ينفذ الوكيل خطوة محفوفة بالمخاطر، حتى تبقى التكلفة وقابلية التراجع والكمون تحت السيطرة.
2. **تصنيف المخاطر**، بحيث يتم تنفيذ الإجراءات منخفضة المخاطر تلقائيًا، وتتم الموافقة على الإجراءات متوسطة المخاطر دفعة واحدة، وتقتصر الإجراءات عالية المخاطر على تدخل بشري.
3. **سجل تدقيق مع حلقة مراجعة**، بحيث يتم تسجيل كل قرار بوابة كـ JSONL، ويرجع الرفض مع إعادة تحفيز الوكيل مع سبب منظم بدلاً من مجرد طباعة `Revising...`.

يقوم هذا الدفتر ببناء كل هذه الميزات على نفس الأساسيات المستخدمة في `06-system-message-framework.ipynb`. يتم تشغيله من البداية للنهاية في وضع `DEMO_MODE = True` (لا حاجة لإدخال تفاعلي) أو مع مطالبات `input()` حقيقية عند `DEMO_MODE = False`. ملاحظة: في وضع العرض، يتم برمجة إعادة المحاولة للهدف الثالث بحيث تكون آلية الحلقة مرئية بشكل كامل. التصنيف المعتمد على المراجعة الحقيقية يتطلب `DEMO_MODE = False` ومشغلًا بشريًا.

**خارج النطاق (يتم التعامل معه في دروس أخرى):** المصادقة والتحكم في الوصول (تهديد الدرس 06 في الملف التعريفي)، وسيط استدعاء الأدوات (درس 14 الغوص العميق في MAF)، أنماط النقاش متعددة الوكلاء.

In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

token = os.environ.get("GITHUB_TOKEN", "")
if not token:
    raise RuntimeError(
        "GITHUB_TOKEN environment variable is not set. This notebook needs "
        "a GitHub PAT with 'Models: Read-only' permission. Set GITHUB_TOKEN "
        "in your environment or a local .env file before running."
    )

endpoint = "https://models.github.ai/inference"
model_name = "gpt-4o"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


## النمط 1: بوابة قبل الفعل

مقتطف HITL في ملف README يستدعي الوكيل أولاً، ثم يطلب من المستخدم الموافقة على النتيجة. هذا هو تدفق **بعد الفعل**. لقد نفذ الوكيل بالفعل، لذلك تم دفع تكلفة استدعاء نموذج اللغة الكبير، وأي تأثير جانبي (مثل إرسال بريد إلكتروني، كتابة صف في قاعدة البيانات، نشر تعليق) قد حدث بالفعل.

تدفق **قبل الفعل** يُدرج البوابة قبل أن ينفذ الوكيل الخطوة الخطرة. يقترح الوكيل الإجراء، وتقرر البوابة ما إذا كانت ستنفذه، ولا يحدث التأثير الجانبي إلا عند الموافقة.

| الجانب | الموافقة بعد الفعل (مقتطف README) | بوابة قبل الفعل (هذا الدفتر) |
|---|---|---|
| متى تتم الموافقة؟ | بعد أن ينتج الوكيل النتيجة | قبل تنفيذ أي تأثير جانبي |
| تكلفة نموذج اللغة بعد الرفض | تم دفعها بالفعل | تُدفع فقط مقابل الاقتراح، وليس الإجراء |
| التأثيرات الجانبية غير القابلة للعكس | ممكنة (الإجراء حدث بالفعل) | ممنوعة |
| وضوح التدقيق | الموافقة عبارة عن عبارة طباعة | الموافقة هي سجل JSON مع الطابع الزمني، الإجراء، السبب |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## النمط 2: تصنيف المخاطر

ليس كل إجراء يحتاج إلى موافقة بشرية. البحث بالقراءة فقط مقابل واجهة برمجة تطبيقات عامة له مخاطرات مختلفة عن إرسال بريد إلكتروني لعميل. التعامل مع الاثنين بنفس الطريقة يهدر انتباه المشغل ويبطئ الوكيل.

نموذج بسيط ذو 3 مستويات:

| المستوى | أمثلة | تدفق الموافقة |
|---|---|---|
| `منخفض` (قراءة فقط) | البحث في قاعدة معرفة، البحث عن خيارات الرحلات، جلب صفحة ويب عامة | تنفيذ تلقائي، مسجل للمراجعة |
| `متوسط` (تغيير رخيص) | تخزين نتيجة مؤقتة، زيادة عداد، جدولة تذكير | تنفيذ تلقائي، لكن مع مراجعة مجمعة يومية |
| `مرتفع` (مواجهة خارجية أو غير قابل للعكس) | إرسال بريد إلكتروني، خصم بطاقة، نشر على قناة عامة | التوقف حتى الموافقة البشرية |

هذا هو أحد التصنيفات. تستخدم أنظمة الإنتاج غالبًا مستويات أكثر تفصيلاً (على سبيل المثال، مستويات أذونات AWS IAM، مستويات وصول تعتمد على الدور). النسخة ذات الثلاث مستويات أدناه هي أصغر نسخة مفيدة لوكيل يدمج بين الأفعال التي تقتصر على القراءة والأفعال ذات التأثير الجانبي.

المصنف أدناه يستخدم قواعد بيانات كلمات رئيسية لكي يبقى العرض التوضيحي محدد النتائج ورخيصًا. في نظام إنتاج، ستستبدل هذا بمصنف متعلم أو محرك سياسة.

In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## النمط 3: سجل التدقيق وحلقة المراجعة

العبارة `print("Response approved.")` ليست سجلاً للتدقيق. من أجل الثقة، يجب تسجيل كل قرار من القرارات كحدث منظم يمكنك بعد ذلك استعلامه أو إعادة تشغيله أو ربطه بمراجعة حادث.

مكونان:

1. **JSONL قابل للإضافة فقط.** سطر واحد لكل قرار، مع طابع زمني، وإجراء، ومستوى، وقرار، وسبب. سهل البحث، وسهل الإرسال إلى مخزن سجلات حقيقي لاحقًا.
2. **حلقة مراجعة عند الرفض.** عندما يعيد البوابة القيمة `deny`، يعيد الوكيل طلبه مع سبب الرفض في السياق، حتى يمكن للاقتراح التالي تجنب المشكلة.

In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.complete(
        model=model_name,
        messages=[SystemMessage(content=system), UserMessage(content=user_text)],
    )
    return response.choices[0].message.content.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}


In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## موارد إضافية

تنفذ عدة مشاريع عامة أخرى أشكالًا مختلفة من هذه الأنماط الخاصة بالبشر في الحلقة. قارن بين الطرق لتجد ما يناسب مجموعتك:

- **LangChain** تغليف أدوات البشر في الحلقة ([الوثائق](https://python.langchain.com/docs/integrations/tools/human_tools)): تغليف أدوات يمكن إدراجها توقف التنفيذ لانتظار إدخال بشري.
- **AutoGen** `UserProxyAgent` ([وثائق الإصدار 0.2](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop)؛ أعيد هيكلة هذا في AutoGen الإصدار 0.4+): يستخدم دور الوكيل خصيصًا لتمثيل الإنسان في محادثات متعددة الوكلاء.
- **Semantic Kernel** مرشحات الدالة ([الوثائق](https://learn.microsoft.com/en-us/semantic-kernel/concepts/enterprise-readiness/filters)): وسيط يعمل حول كل نداء دالة، مناسب لمنطق الحصار.

يتعامل كل مشروع مع الأنماط الفرعية الثلاثة بشكل مختلف: LangChain يغلفها كأدوات، AutoGen يستخدم دور وكيل، Semantic Kernel يستخدم مرشحات وسيطة. اقرأ تنفيذًا واحدًا أو اثنين من البداية للنهاية قبل اختيار تصميم لوكيلك الخاص.

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**تنويه**:
تمت ترجمة هذا المستند باستخدام خدمة الترجمة بالذكاء الاصطناعي [Co-op Translator](https://github.com/Azure/co-op-translator). بينما نسعى للدقة، يرجى العلم أن الترجمات الآلية قد تحتوي على أخطاء أو عدم دقة. يجب اعتبار المستند الأصلي بلغته الأصلية المصدر الرسمي والمعتمد. للمعلومات الهامة، يُنصح بالاستعانة بترجمة بشرية محترفة. نحن غير مسؤولين عن أي سوء فهم أو تفسير ناتج عن استخدام هذه الترجمة.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
